<!-- GENERATED by scripts/gen_notebooks.py from _variables.yml. Do not edit this cell. -->
# Module 0 · Pre-work: environment, warm-up and coding agents

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/marketing-statistics-workshop/blob/v2026.10.0/labs/r/00-setup-warmup.ipynb)

**Statistics for Marketing** · Before Day 1 · R · 60-minute lab (optional pre-work)

Check that your environment runs, turn a raw transaction log into a customer table in DuckDB SQL and in pandas (or dplyr), and see how to check code that a coding agent wrote.

**By the end of this notebook you can:**

- Run the environment check on Colab or locally and read what its report says.
- Build a one-row-per-customer table from a transaction log with DuckDB SQL (GROUP BY, MIN, MAX, COUNT, SUM).
- Rebuild the same table in pandas (or dplyr in R) and check that the two agree.
- Count repeat purchases on distinct days, not rows, and explain why the difference matters for later models.
- Verify code written by a coding agent with a checkpoint before you trust it.

**Stack:** R 4.6.1, duckdb, DBI, dplyr

**The decision:** Decide where you will run the labs (Colab, local Docker or AWS) and whether you will do the R notebooks, based on an environment check that actually ran.

**How this notebook works.** Run the cells in order (Runtime > Run all is fine: it stops at the first exercise you have not written). Each exercise has a stub to complete, a folded solution, and a checkpoint that says whether your code passed.

In [ ]:
# Generated by scripts/gen_notebooks.py from _variables.yml: do not edit this cell.
# On Colab: installs the R packages this lab needs (binaries from a dated Posit Public
# Package Manager snapshot) and sources the workshop helpers. Elsewhere: installs nothing.
.mkt_t0 <- Sys.time()
MKTSTATS_REF <- "v2026.10.0"
options(mktstats.ref = MKTSTATS_REF, mktstats.raw = "https://raw.githubusercontent.com/project-delphi/marketing-statistics-workshop")
local({
  cran <- c("jsonlite", "duckdb", "DBI", "dplyr")
  github <- list()
  needs_gsl <- FALSE
  root <- Sys.getenv("MKTSTATS_REPO_ROOT")
  hosted <- nzchar(Sys.getenv("COLAB_RELEASE_TAG")) || nzchar(Sys.getenv("SAGEMAKER_APP_TYPE")) ||
    file.exists("/opt/ml/metadata/resource-metadata.json")
  missing <- setdiff(c(cran, names(github)), rownames(installed.packages()))
  if (nzchar(root) || !hosted) {
    if (length(missing)) {
      warning("Not installed here: ", paste(missing, collapse = ", "),
              ". Use the workshop image (environment/Dockerfile) or install them.", call. = FALSE)
    }
    return(invisible())
  }
  options(
    repos = c(CRAN = "https://p3m.dev/cran/__linux__/noble/2026-10-01"),
    HTTPUserAgent = sprintf(
      "R/%s R (%s)", getRversion(),
      paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"])
    ),
    timeout = 900
  )
  if (needs_gsl && !any(grepl("libgsl\\.so", system("ldconfig -p", intern = TRUE)))) {
    message("Installing the GSL system library ...")
    system("apt-get install -y -qq libgsl27 > /tmp/mktstats-apt.log 2>&1")
  }
  need <- setdiff(cran, rownames(installed.packages()))
  if (length(need)) install.packages(need)
  for (name in names(github)) {
    if (!name %in% rownames(installed.packages())) {
      archive <- sprintf("https://github.com/%s/archive/%s.tar.gz", github[[name]][1],
                         github[[name]][2])
      install.packages(archive, repos = NULL, type = "source")
    }
  }
  still <- setdiff(c(cran, names(github)), rownames(installed.packages()))
  if (length(still)) {
    stop("Could not install: ", paste(still, collapse = ", "),
         ". Rerun this cell; if it fails again, tell the instructor.", call. = FALSE)
  }
})
local({
  root <- Sys.getenv("MKTSTATS_REPO_ROOT")
  if (!nzchar(root)) {  # a notebook opened inside a checkout of the repository
    here <- normalizePath(getwd())
    for (i in 1:4) {
      if (file.exists(file.path(here, "R", "mktstats.R"))) { root <- here; break }
      here <- dirname(here)
    }
  }
  helpers <- if (nzchar(root)) file.path(root, "R", "mktstats.R") else
    sprintf("%s/%s/R/mktstats.R", getOption("mktstats.raw"), MKTSTATS_REF)
  source(helpers)
})
.mkt_secs <- as.numeric(difftime(Sys.time(), .mkt_t0, units = "secs"))
Sys.setenv(MKTSTATS_INSTALL_SECONDS = sprintf("%.1f", .mkt_secs))
.mkt_where <- Sys.getenv("COLAB_RELEASE_TAG")
cat("Runtime:", if (nzchar(.mkt_where)) paste("colab", .mkt_where) else "local", "·",
    R.version.string, "·", parallel::detectCores(), "CPUs · setup took",
    Sys.getenv("MKTSTATS_INSTALL_SECONDS"), "s\n")

In [ ]:
# Generated by scripts/gen_notebooks.py from _variables.yml: do not edit this cell.
# WORKED_EXAMPLE <- TRUE runs every reference solution in place of your code: a
# complete worked example, not a sign that you did the exercises. QUICK <- TRUE
# shortens long steps (sampling, data size).
WORKED_EXAMPLE <- FALSE
QUICK <- FALSE
.ws_start(
  notebook = "r/00-setup-warmup",
  content_sha = "e6216872043093cf",
  deps_sha = "cb6b0363735a9c97",
  ref = "v2026.10.0",
  exercises = list(`1` = c("customer_table_sql"), `2` = c("customer_table_dplyr"), `3` = c("repeat_purchases"))
)

# Part A · Your runtime

A *runtime* is the machine your notebook runs on. On Colab it is a virtual machine that
Google starts for you and deletes when you disconnect, so anything you install or download
lasts only for the session. The cell above installed what this lab needs (on Colab) and
printed where you are running.

**Predict.** How many CPUs does this runtime have: 1, 2, 4 or 8? And how much memory: under
8 GB, 8 to 16 GB, or more? Write both guesses down before you run the next cell.

In [ ]:
suppressPackageStartupMessages({
  library(DBI)
  library(dplyr)
})
memory_gb <- tryCatch({
  line <- grep("^MemTotal", readLines("/proc/meminfo"), value = TRUE)
  round(as.numeric(gsub("[^0-9]", "", line)) * 1024 / 1e9, 1)
}, error = function(e) NA, warning = function(w) NA)
cat("R:            ", R.version.string, "\n")
cat("Platform:     ", R.version$platform, "\n")
cat("CPUs:         ", parallel::detectCores(), "\n")
cat("Memory (GB):  ", memory_gb, "\n")
cat("Colab release:", Sys.getenv("COLAB_RELEASE_TAG", "(not on Colab)"), "\n")
for (p in c("duckdb", "DBI", "dplyr")) cat(sprintf("%-14s%s\n", p, as.character(packageVersion(p))))
cat("QUICK:        ", QUICK, "(it has no effect in this lab)\n")

**Explain.** Compare with your guesses. The CPU count and memory depend on the runtime type
you choose (on Colab: Runtime > Change runtime type), so keep a note of them: later labs
report run times, and a time means little without the machine it ran on. Google publishes
the software each Colab runtime ships with (operating system, Python and R versions,
preinstalled packages) in the
[googlecolab/backend-info](https://github.com/googlecolab/backend-info) repository. The
workshop's R labs install their packages from a dated snapshot of the Posit Public Package
Manager, the same snapshot its Docker image uses, so CI and Colab run the same versions.

# Part B · From a transaction log to a customer table

Most customer analytics starts from a *transaction log*: one row per purchase. Here it is
the CDNOW log that ships with PyMC-Marketing's example data. CDNOW was an online CD
retailer. The columns:

| column | meaning |
|---|---|
| `_id`, `id` | two identifiers of the customer, one-to-one; we use `id` |
| `date` | the purchase date as an integer, `YYYYMMDD` (for example `19970118`) |
| `cds_bought` | number of CDs in the purchase |
| `spent` | amount paid, in dollars |

The next cell downloads the file once, checks it against a known SHA-256 hash (so you know
you have the same bytes as everyone else), and caches it.

In [ ]:
# TODO(mktstats data loaders): switch to the shared CDNOW loader once the package has one.
CDNOW_URL <- "https://raw.githubusercontent.com/pymc-labs/pymc-marketing/main/data/cdnow_transactions.csv"
CDNOW_SHA256 <- "7311ae0fb676e9f454bba310b10bc769b1a6838aeec09bd00175eb49481dddac"

load_cdnow <- function(url = CDNOW_URL, sha256 = CDNOW_SHA256) {
  cache <- Sys.getenv("MKTSTATS_CACHE", file.path(path.expand("~"), ".cache", "mktstats"))
  path <- file.path(cache, "cdnow_transactions.csv")
  good <- function() file.exists(path) && identical(unname(tools::sha256sum(path)), sha256)
  if (!good()) {
    dir.create(cache, recursive = TRUE, showWarnings = FALSE)
    for (attempt in 1:3) {
      done <- tryCatch({
        utils::download.file(url, path, quiet = TRUE, mode = "wb")
        TRUE
      }, error = function(e) FALSE, warning = function(w) FALSE)
      if (done) break
      Sys.sleep(2 * attempt)
    }
    if (!good()) {
      unlink(path)
      stop("Could not download the CDNOW file with the expected SHA-256 hash. ",
           "Check the network and rerun; if it fails again, tell the instructor.", call. = FALSE)
    }
  }
  utils::read.csv(path, check.names = FALSE)  # keep the column name `_id`
}

tx_raw <- load_cdnow()
con <- suppressMessages(dbConnect(duckdb::duckdb()))  # quiet the storage notice
duckdb::duckdb_register(con, "transactions", tx_raw)
cat(format(nrow(tx_raw), big.mark = ","), "transactions,",
    format(length(unique(tx_raw$id)), big.mark = ","), "customers\n")
head(tx_raw)

DuckDB is an analytical database that runs inside this notebook: no server, and it can query
an R data frame directly. `duckdb_register(con, "transactions", tx_raw)` made the data frame
visible to SQL as a table named `transactions`. Try it:

In [ ]:
dbGetQuery(con, "SELECT COUNT(*) AS n_transactions, SUM(spent) AS total_spend FROM transactions")

## Exercise 1 · The customer table in SQL (12 minutes)

**Predict.** The log has one row per transaction, and the cell above printed how many
customers it covers. What share of those customers bought only once: under 25%, 25 to 50%,
or over 50%? Write your guess down; the cell after Exercise 3 prints the answer.

**Task.** Write `customer_table_sql(con)`: a DuckDB query on `transactions` that returns one
row per customer with these columns:

- `customer_id` (the `id` column)
- `first_purchase` and `last_purchase`: dates, not `YYYYMMDD` integers. In DuckDB,
  `strptime(CAST(date AS VARCHAR), '%Y%m%d')::DATE` turns `19970118` into a date.
- `n_purchases`: the number of transactions
- `total_spend`: the sum of `spent`

Return a data frame: `dbGetQuery(con, query)`.

In [ ]:
customer_table_sql <- function(con) {
  # TODO 1: write the query, run it with dbGetQuery(con, query)
  stop("TODO 1")
}

**Solution 1 — try it yourself first.** The next cell is the reference solution.

In [ ]:
#@title Solution 1 — try it yourself first { display-mode: "form" }
solution(1, "customer_table_sql", function(con) {
  dbGetQuery(con, "
    SELECT
      id AS customer_id,
      MIN(strptime(CAST(date AS VARCHAR), '%Y%m%d'))::DATE AS first_purchase,
      MAX(strptime(CAST(date AS VARCHAR), '%Y%m%d'))::DATE AS last_purchase,
      COUNT(*) AS n_purchases,
      SUM(spent) AS total_spend
    FROM transactions
    GROUP BY id
    ORDER BY customer_id
  ")
})

**Explain.** The table has exactly one row per customer, the count printed above, because
`GROUP BY id` collapses each customer's rows into one; every other column must then be an
aggregate (`MIN`, `MAX`, `COUNT`, `SUM`). Converting the date before taking `MIN` and `MAX`
matters little here (`YYYYMMDD` integers sort in date order) but matters as soon as you
subtract dates. Your guess about one-time buyers is checked after Exercise 3.

<details><summary>Why convert the dates at all?</summary>

Recency and customer age, the inputs of the models in Modules 1 to 4, are differences between
dates. `19970201 - 19970131` is 70, not 1: integer dates give nonsense the moment you do
arithmetic with them.
</details>

In [ ]:
checkpoint(1, {
  customers_sql <- customer_table_sql(con)
  check_columns(customers_sql, c("customer_id", "first_purchase", "last_purchase", "n_purchases", "total_spend"),
                name = "customer_table_sql(con)")
  n_customers <- length(unique(tx_raw$id))
  if (nrow(customers_sql) != n_customers || anyDuplicated(customers_sql$customer_id)) {
    stop(sprintf("Expected one row per customer (%d), got %d rows. GROUP BY the customer id.",
                 n_customers, nrow(customers_sql)), call. = FALSE)
  }
  if (!inherits(customers_sql$first_purchase, "Date")) {
    stop("first_purchase should be a date, not an integer like 19970118: convert it with ",
         "strptime(CAST(date AS VARCHAR), '%Y%m%d')::DATE.", call. = FALSE)
  }
  if (any(customers_sql$first_purchase > customers_sql$last_purchase)) {
    stop("Some customers' first purchase comes after their last: check MIN and MAX.", call. = FALSE)
  }
  check_close(sum(customers_sql$n_purchases), nrow(tx_raw), abs = 0,
              name = "the number of purchases summed over customers")
  check_close(sum(customers_sql$total_spend), sum(tx_raw$spent), rel = 1e-9,
              name = "total spend summed over customers")
})

## Exercise 2 · The same table in dplyr (12 minutes)

Two independent implementations that agree are the cheapest test you will ever write.

**Predict.** Which step is most likely to make a dplyr version disagree with the SQL one:
the date conversion, the purchase count, or the spend total? Pick one and write it down.

**Task.** Write `customer_table_dplyr(tx)` returning the same columns as Exercise 1, from the
data frame `tx` (the raw log). `as.Date(as.character(date), format = "%Y%m%d")` converts the
dates; then `group_by()` the customer and `summarise()`.

In [ ]:
customer_table_dplyr <- function(tx) {
  # TODO 2: convert the dates, group by customer, summarise
  stop("TODO 2")
}

**Solution 2 — try it yourself first.** The next cell is the reference solution.

In [ ]:
#@title Solution 2 — try it yourself first { display-mode: "form" }
solution(2, "customer_table_dplyr", function(tx) {
  tx |>
    mutate(date = as.Date(as.character(date), format = "%Y%m%d")) |>
    group_by(customer_id = id) |>
    summarise(
      first_purchase = min(date),
      last_purchase = max(date),
      n_purchases = n(),
      total_spend = sum(spent),
      .groups = "drop"
    ) |>
    arrange(customer_id)
})

**Explain.** Compare with your pick. If the checkpoint failed on your first try, which column
did its message name, and what made that step differ from SQL? `summarise()` mirrors the SQL
`SELECT` list one to one, which makes the two easy to compare by eye. `n()` counts rows, as
`COUNT(*)` does.

<details><summary>What the checkpoint compares</summary>

It matches the two tables on `customer_id` and compares every shared column: dates exactly,
counts and spend within a tiny tolerance (floating-point sums can differ in the last digit
depending on the order of addition).
</details>

In [ ]:
checkpoint(2, {
  customers_dplyr <- customer_table_dplyr(tx_raw)
  check_frames_agree(customers_sql, customers_dplyr, on = "customer_id",
                     names = c("the SQL table", "the dplyr table"))
})

## Exercise 3 · Repeat purchases count days, not rows (10 minutes)

The models from Module 2 on describe *repeat purchases*: purchases after the first. They
count purchase occasions, not rows, so two transactions by the same customer on the same day
are one occasion. Then `repeat_purchases = (number of distinct purchase days) - 1`.

**Predict.** Do any CDNOW customers have two transactions on the same day? For how many
customers will counting rows instead of days change the number of repeat purchases: none,
under 100, or 100 or more? Write your guess down.

**Task.** Write `repeat_purchases(tx)` with dplyr, returning one row per customer with
`customer_id`, `purchase_days` (distinct dates) and `repeat_purchases`.

In [ ]:
repeat_purchases <- function(tx) {
  # TODO 3: count distinct purchase days per customer, then subtract one
  stop("TODO 3")
}

**Solution 3 — try it yourself first.** The next cell is the reference solution.

In [ ]:
#@title Solution 3 — try it yourself first { display-mode: "form" }
solution(3, "repeat_purchases", function(tx) {
  tx |>
    group_by(customer_id = id) |>
    summarise(purchase_days = n_distinct(date), .groups = "drop") |>
    mutate(repeat_purchases = purchase_days - 1L)
})

**Explain.** `n_distinct()` counts distinct values, so same-day transactions collapse into
one occasion. A customer who bought once has zero repeat purchases.

<details><summary>Why the checkpoint uses a tiny made-up log</summary>

Customer 1 below buys twice on 1 January and once on 1 March: one repeat purchase, although
there are three rows. A log built to contain the tricky case tests the definition directly;
the full CDNOW log then checks your function against an independent SQL count.
</details>

In [ ]:
checkpoint(3, {
  toy <- data.frame(
    `_id` = c(10, 10, 10, 20), id = c(1, 1, 1, 2),
    date = c(19970101, 19970101, 19970301, 19970105),
    cds_bought = c(1, 2, 1, 1), spent = c(10, 20, 5, 7),
    check.names = FALSE
  )
  got <- repeat_purchases(toy)
  r1 <- got$repeat_purchases[got$customer_id == 1]
  r2 <- got$repeat_purchases[got$customer_id == 2]
  if (!isTRUE(all.equal(as.numeric(r1), 1))) {
    stop("Customer 1 bought twice on 1997-01-01 and once on 1997-03-01: one repeat purchase ",
         "(same-day purchases count once). Your function gives ", format(r1), ".", call. = FALSE)
  }
  if (!isTRUE(all.equal(as.numeric(r2), 0))) {
    stop("Customer 2 bought once: zero repeat purchases, not ", format(r2), ".", call. = FALSE)
  }
  expected <- dbGetQuery(con, paste("SELECT id AS customer_id, COUNT(DISTINCT date) - 1 AS repeat_purchases",
                                    "FROM transactions GROUP BY id"))
  check_frames_agree(as.data.frame(repeat_purchases(tx_raw))[c("customer_id", "repeat_purchases")], expected,
                     on = "customer_id", names = c("repeat_purchases(tx_raw)", "an independent SQL count"))
})

Compare with both predictions: the share of customers who bought only once (Exercise 1) and
the number whose count changes (Exercise 3). Counting rows would give those customers extra
"repeat purchases" on days they already bought: which input of the Module 2 models
(frequency, recency or age) would that inflate, and would it make them look more or less
loyal than they are?

In [ ]:
repeats <- repeat_purchases(tx_raw)
both <- merge(customers_sql, repeats, by = "customer_id")
cat(sprintf("Customers who bought only once: %.1f%%\n", 100 * mean(both$repeat_purchases == 0)))
cat("Customers whose count changes if you count rows instead of days:",
    sum(both$n_purchases - 1 != both$repeat_purchases), "\n")

# Part C · Using a coding agent to scaffold and check an analysis

A *coding agent* is an AI assistant that can write code, and often run it, in your project.
It can draft queries like the ones above from a description of the tables. Treat what it
writes as a colleague's first draft: useful, and unverified until checked.

1. **Give it the definitions, not just the column names.** "Repeat purchases" was ambiguous
   until Exercise 3 pinned it down (days, not rows). Ambiguity is where a plausible-looking
   query goes wrong.
2. **Ask for an independent check, not only the answer.** A second implementation in another
   tool (Exercise 2) and invariants that must hold (one row per customer, totals that match
   the raw log) catch errors that reading the code misses.
3. **Run the checks yourself and read the code.** An agent saying "this works" is not
   evidence; a passing checkpoint is.
4. **Mind the data.** Do not paste customer records into a tool your organization has not
   approved for them. Describing the schema, or sharing a small made-up sample like the one
   in checkpoint 3, is usually enough.

A prompt in this spirit:

> Table `transactions(_id, id, date, cds_bought, spent)`: `id` is the customer, `date` is an
> integer `YYYYMMDD`. Write a DuckDB query with one row per customer: first and last purchase
> dates (as dates), number of transactions, total spend. Then write a dplyr version and a
> test that the two agree on every customer.

## Decision · Sign off the customer table

The pre-work ends in two decisions, both taken from evidence this notebook produced: is the
customer table fit to hand to Module 1, and where will you run the labs?

**Number.** The runtime report in Part A (CPUs, memory, versions) and the printout below:
whether the table's purchases and spend match the raw log, and which checkpoints passed.

**Rule.** Hand the table on only if every checkpoint passed and the totals match the log.
Run the labs on Colab if this notebook ran here with every checkpoint passing; use the
workshop's Docker image (see the Setup page of the workshop site; the site is linked at the
end of this notebook) if the Colab install failed twice or your organization blocks Colab.
Do the R notebooks of Modules 3, 7 and 11 if this R notebook ran with every checkpoint
passing; if it failed and you have no Docker, follow those modules in Python and from the
module pages.

In [ ]:
cat("Customers:", format(nrow(both), big.mark = ","), "\n")
cat("Purchases:", format(sum(both$n_purchases), big.mark = ","), "(log:", format(nrow(tx_raw), big.mark = ","), ")\n")
cat(sprintf("Spend: $%s (log: $%s)\n", format(round(sum(both$total_spend), 2), big.mark = ",", nsmall = 2),
            format(round(sum(tx_raw$spent), 2), big.mark = ",", nsmall = 2)))
cat("Window:", format(min(both$first_purchase)), "to", format(max(both$last_purchase)), "\n")
passed <- names(Filter(function(r) isTRUE(r$passed), .ws$results))
cat("Checkpoints passed:", if (length(passed)) paste(passed, collapse = ", ") else "none", "\n")

**Recommendation.** Write two sentences. First: would you hand this table to the modeling
team, yes or no, and which checks your answer rests on? Second: where will you run the labs
(Colab, Docker or AWS), will you do the R notebooks, and which result above does that rest
on? The site's FAQ covers the usual setup problems.

Your sentences: ________________________________________________

## Stretch (optional) · Customers by first-purchase month

Write a DuckDB query that counts customers by the month of their first purchase and gives
each cohort's total spend over the whole window. Which month brought the most valuable
customers per head?

<details><summary>One way to write it</summary>

```sql
WITH firsts AS (
    SELECT id, MIN(strptime(CAST(date AS VARCHAR), '%Y%m%d'))::DATE AS first_purchase,
           SUM(spent) AS spend
    FROM transactions GROUP BY id
)
SELECT date_trunc('month', first_purchase) AS cohort, COUNT(*) AS customers,
       SUM(spend) AS spend, SUM(spend) / COUNT(*) AS spend_per_customer
FROM firsts GROUP BY cohort ORDER BY cohort
```
</details>

In [ ]:
# Generated by scripts/gen_notebooks.py from _variables.yml: do not edit this cell.
# Which checkpoints passed, and on whose code; then the run record (also saved as run_record.json).
workshop_summary()
run_record()

<!-- GENERATED by scripts/gen_notebooks.py from _variables.yml. Do not edit this cell. -->
---
**Next:** [Module 3 · Covariates in CLVTools, and BTYD versus machine learning](https://colab.research.google.com/github/project-delphi/marketing-statistics-workshop/blob/v2026.10.0/labs/r/03-clvtools-covariates.ipynb) (R)

[Workshop site](https://project-delphi.github.io/marketing-statistics-workshop/) · [All notebooks](https://project-delphi.github.io/marketing-statistics-workshop/notebooks.html) · [Source](https://github.com/project-delphi/marketing-statistics-workshop)

Genial Labs · text CC BY 4.0, code MIT